In [1]:
import re
import numpy as np
import pandas as pd
from bottleneck_function import *

In [2]:
def S(x) -> str:
    return str(x).strip()

def build_groups_from_merged_header(df_raw: pd.DataFrame): 
    # Table-S7 has a merged/group header row stored in df_raw.columns. Unnamed:*' columns inherit the last non-Unnamed header. Returns a list 'groups' aligned with df_raw.columns.
    cols = list(df_raw.columns)
    groups = []
    cur = None
    for c in cols:
        if not str(c).startswith("Unnamed"):
            cur = c
        groups.append(cur)
    return groups

def find_col_by_sample_and_group(df_raw, sample_names, groups, sample_label, group_keyword):
    # Find the *dataframe column name* whose sample-name (row 0) matches sample_label AND whose merged header group contains group_keyword.
    gkw = group_keyword.lower()
    idxs = [i for i, nm in enumerate(sample_names) if S(nm) == sample_label and (groups[i] and gkw in str(groups[i]).lower())]
    if not idxs:
        raise ValueError(f"Cannot find sample '{sample_label}' inside group containing '{group_keyword}'.")
    return df_raw.columns[idxs[0]]

def idxs_by_regex_and_group(sample_names, groups, pattern, group_keyword):
    #Return column indices i where sample_names[i] matches regex pattern (fullmatch) and the merged header group contains group_keyword.
    pat = re.compile(pattern)
    gkw = group_keyword.lower()
    out = []
    for i, nm in enumerate(sample_names):
        if pat.fullmatch(S(nm)) and (groups[i] and gkw in str(groups[i]).lower()):
            out.append(i)
    return out

def get_counts_vector(sgrna_df: pd.DataFrame, colname) -> np.ndarray: #Convert one sample column to an integer count vector."""
    return (pd.to_numeric(sgrna_df[colname], errors="coerce").fillna(0).round().astype(int).to_numpy())

In [3]:
# load Table-S7 (raw, like notebook)
df_raw = pd.read_csv("Table-S7-sgRNA-Raw-Counts.csv")          # keeps the merged headers in df_raw.columns
sample_names = df_raw.iloc[0].astype(str).tolist()   # the *second* CSV row (sample names)

# data rows (sgRNA001..), drop fully-empty trailing row if present
sgrna_df = df_raw.iloc[1:].copy()
sgrna_df = sgrna_df.dropna(how="all")

groups = build_groups_from_merged_header(df_raw)

In [4]:
# ---------------------------- 24 hpi ----------------------------
PRE24_GROUP = "preinfection samples for 24 hpi"
H24_GROUP   = "24 hpi of murine pneumonia model"

donor_pre1_24_col = find_col_by_sample_and_group(df_raw, sample_names, groups, "Pre1", PRE24_GROUP)
donor_pre2_24_col = find_col_by_sample_and_group(df_raw, sample_names, groups, "Pre2", PRE24_GROUP)

donor_pre1_24 = get_counts_vector(sgrna_df, donor_pre1_24_col)  # noDox donor (as in notebook)
donor_pre2_24 = get_counts_vector(sgrna_df, donor_pre2_24_col)  # +dox donor (as in notebook)

nodox_24_idxs = idxs_by_regex_and_group(sample_names, groups, r"Mice_noDox_\d+",   H24_GROUP)
dox_24_idxs   = idxs_by_regex_and_group(sample_names, groups, r"Mice_plusDox_\d+", H24_GROUP)

records24 = []
for i in nodox_24_idxs:
    nm = S(sample_names[i])
    mouse = int(nm.split("_")[-1])
    col = df_raw.columns[i]
    x = get_counts_vector(sgrna_df, col)
    nb_hat = bottleneck_from_two_timepoints(donor_pre1_24, x)
    records24.append(("noDox", mouse, "Lung", nb_hat, nm))

for i in dox_24_idxs:
    nm = S(sample_names[i])
    mouse = int(nm.split("_")[-1])
    col = df_raw.columns[i]
    x = get_counts_vector(sgrna_df, col)
    nb_hat = bottleneck_from_two_timepoints(donor_pre2_24, x)
    records24.append(("Dox", mouse, "Lung", nb_hat, nm))

res24_long = pd.DataFrame(records24, columns=["condition", "mouse", "tissue", "Nb_hat", "sample"])

export24 = (res24_long.pivot_table(index=["condition", "mouse"], columns="tissue", values="Nb_hat", aggfunc="first").reset_index().rename_axis(None, axis=1))
export24["hpi"] = 24
export24["label"] = export24.apply(lambda r: f"#{int(r['mouse'])}" if r["condition"] == "noDox" else f"#{int(r['mouse'])}+dox", axis=1)

if "Blood" not in export24.columns:
    export24["Blood"] = np.nan
export24 = export24.rename(columns={"Lung": "lung", "Blood": "blood"})[["hpi","condition","mouse","label","lung","blood"]]
export24.to_csv("Leonard_Nb_24hpi.csv", index=False)

In [5]:
# ---------------------------- 48 hpi ----------------------------
PRE48_GROUP = "preinfection samples for 48 hpi"
H48_GROUP   = "48 hpi of murine pneumonia model"

donor_pre1_48_col = find_col_by_sample_and_group(df_raw, sample_names, groups, "Pre1", PRE48_GROUP)
donor_pre2_48_col = find_col_by_sample_and_group(df_raw, sample_names, groups, "Pre2", PRE48_GROUP)

donor_pre1_48 = get_counts_vector(sgrna_df, donor_pre1_48_col)
donor_pre2_48 = get_counts_vector(sgrna_df, donor_pre2_48_col)

# notebook mapping: Pre1 for Dox, Pre2 for noDox (swap if you want the opposite)
DONOR48 = {"Dox": donor_pre1_48, "noDox": donor_pre2_48}

idxs_48 = [i for i in range(len(sample_names)) if groups[i] and H48_GROUP.lower() in str(groups[i]).lower()]

pat48 = re.compile(r"Mice_(noDox|Dox)-(\d+)-(Blood|Lung)$")
records48 = []
for i in idxs_48:
    nm = S(sample_names[i])
    m = pat48.fullmatch(nm)
    if not m:
        continue
    cond, mouse, tissue = m.group(1), int(m.group(2)), m.group(3)
    col = df_raw.columns[i]
    x = get_counts_vector(sgrna_df, col)
    nb_hat = bottleneck_from_two_timepoints(DONOR48[cond], x)
    records48.append((cond, mouse, tissue, nb_hat, nm))

res48_long = pd.DataFrame(records48, columns=["condition", "mouse", "tissue", "Nb_hat", "sample"])

export48 = (res48_long.pivot_table(index=["condition", "mouse"], columns="tissue", values="Nb_hat", aggfunc="first").reset_index().rename_axis(None, axis=1))
export48["hpi"] = 48
export48["label"] = export48.apply(lambda r: f"#{int(r['mouse'])}" if r["condition"] == "noDox" else f"#{int(r['mouse'])}+dox",axis=1)

for t in ["Lung", "Blood"]:
    if t not in export48.columns:
        export48[t] = np.nan
export48 = export48.rename(columns={"Lung": "lung", "Blood": "blood"})[["hpi","condition","mouse","label","lung","blood"]]
export48.to_csv("Leonard_Nb_48hpi.csv", index=False)